### References

*   [https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876](https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876)
*   [https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo](https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo)
*   [https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/](https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/)
*   https://www.kaggle.com/code/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch
*   [https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference](https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference)
*   https://www.kaggle.com/code/neibyr/30-min-just-use-semantic-search-qwen3-emb-0-6b
*   https://www.kaggle.com/code/datafan07/jigsaw-speed-run-10-min-triplet-and-faiss
*   https://www.kaggle.com/code/nahidhossainredom/deberta-v3-base-3-epochs-lb-0-906

In [1]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

Using Python 3.11.13 environment at: /usr
Resolved 168 packages in 559ms                                       
   Building deepspeed==0.17.4                                          
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                                  
   Building deepspeed==0.17.4                   

In [2]:
# !uv pip install transformers==4.45.2

In [3]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/qwen2.5/transformers/0.5b-instruct-gptq-int4/1"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

Writing constants.py


In [4]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT
import random, numpy as np
random.seed(42)
np.random.seed(42)


def build_prompt(row):
    return f"""
{BASE_PROMPT}

Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
{COMPLETE_PHRASE} Yes

2) {row["negative_example"]}
{COMPLETE_PHRASE} No

---
Comment: {row["body"]}
{COMPLETE_PHRASE}"""


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv").sample(frac=0.5, random_state=42).reset_index(drop=True)

    flatten = []

    # ---------- 处理训练集 ----------
    train_df = train_dataset[["body", "rule", "subreddit", "rule_violation",
                              "positive_example_1","positive_example_2",
                              "negative_example_1","negative_example_2"]].copy()

    # 随机选 positive_example 和 negative_example
    train_df["positive_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["positive_example_1"],
        train_df["positive_example_2"]
    )
    train_df["negative_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["negative_example_1"],
        train_df["negative_example_2"]
    )

    # 删除原来的候选列
    train_df.drop(columns=["positive_example_1","positive_example_2",
                           "negative_example_1","negative_example_2"], inplace=True)

    flatten.append(train_df)

    # ---------- 处理测试集 ----------
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[["rule","subreddit",
                                        "positive_example_1","positive_example_2",
                                        "negative_example_1","negative_example_2"]].copy()

            if violation_type == "positive":
                # body 用当前 positive_example
                body_col = f"positive_example_{i}"
                other_positive_col = f"positive_example_{3-i}"  # 另一个 positive
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["positive_example"] = sub_dataset[other_positive_col]
                # negative_example 随机选
                sub_dataset["negative_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["negative_example_1"],
                    sub_dataset["negative_example_2"]
                )
                sub_dataset["rule_violation"] = 1

            else:  # violation_type == "negative"
                body_col = f"negative_example_{i}"
                other_negative_col = f"negative_example_{3-i}"
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["negative_example"] = sub_dataset[other_negative_col]
                sub_dataset["positive_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["positive_example_1"],
                    sub_dataset["positive_example_2"]
                )
                sub_dataset["rule_violation"] = 0

            # 删除原来的候选列
            sub_dataset.drop(columns=["positive_example_1","positive_example_2",
                                      "negative_example_1","negative_example_2"], inplace=True)

            flatten.append(sub_dataset)

    # 合并所有 DataFrame
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)

    return dataframe



def build_dataset(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]
    if "rule_violation" in dataframe:
        dataframe["completion"] = dataframe["rule_violation"].map(
            {
                1: POSITIVE_ANSWER,
                0: NEGATIVE_ANSWER,
            }
        )
        columns.append("completion")

    dataframe = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe)
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)
    return dataset

Writing utils.py


In [ ]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH, LORA_PATH


def main():
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        num_train_epochs=1,

        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,

        optim="paged_adamw_8bit",
        learning_rate=1e-4, #keep high, lora usually likes high.
        weight_decay=0.01,
        max_grad_norm=1.0,

        lr_scheduler_type="cosine",
        warmup_ratio=0.03,

        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),
        dataloader_pin_memory=True,

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        save_strategy="no",
        report_to="none",

        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        BASE_MODEL_PATH,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )

    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

Writing train.py


In [6]:
%%writefile inference.py
import os
os.environ["VLLM_USE_V1"] = "0"

import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import BASE_MODEL_PATH, LORA_PATH, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER
import random
import multiprocessing as mp


def run_inference_on_device(df_slice):
    """在当前进程可见的 GPU 上跑 vLLM 推理"""
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        quantization="gptq",
        tensor_parallel_size=1,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2836,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )

    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])

    test_dataset = build_dataset(df_slice)
    texts = test_dataset["prompt"]

    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )

    log_probs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    predictions = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    predictions["row_id"] = df_slice["row_id"].values
    return predictions


def worker(device_id, df_slice, return_dict):
    # 限制该进程只看到一张 GPU
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] Running on GPU {device_id}, data size={len(df_slice)}")

    preds = run_inference_on_device(df_slice)
    return_dict[device_id] = preds


def main():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")

    # 随机选择例子
    test_dataframe["positive_example"] = test_dataframe.apply(
        lambda row: random.choice([row["positive_example_1"], row["positive_example_2"]]),
        axis=1
    )
    test_dataframe["negative_example"] = test_dataframe.apply(
        lambda row: random.choice([row["negative_example_1"], row["negative_example_2"]]),
        axis=1
    )
    test_dataframe = test_dataframe.drop(
        columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"],
        errors="ignore"
    )

    # 切分数据
    mid = len(test_dataframe) // 2
    df0 = test_dataframe.iloc[:mid].reset_index(drop=True)
    df1 = test_dataframe.iloc[mid:].reset_index(drop=True)

    manager = mp.Manager()
    return_dict = manager.dict()

    # 两个进程并行
    p0 = mp.Process(target=worker, args=(0, df0, return_dict))
    p1 = mp.Process(target=worker, args=(1, df1, return_dict))
    p0.start()
    p1.start()
    p0.join()
    p1.join()

    # 合并结果
    predictions = pd.concat([return_dict[0], return_dict[1]], ignore_index=True)

    # 构建 submission
    submission = predictions[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    rq = submission['rule_violation'].rank(method='average') / (len(submission) + 1)
    submission['rule_violation'] = rq

    submission.to_csv("submission_qwen.csv", index=False)
    print("✅ Saved submission_qwen.csv")


if __name__ == "__main__":
    main()

Writing inference.py


In [ ]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_batch_size: 64
  train_micro_batch_size_per_gpu: 4

  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false

  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5

  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false

  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1

distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

Writing accelerate_config.yaml


In [8]:
!accelerate launch --config_file accelerate_config.yaml train.py

In [9]:
!python inference.py

In [10]:
import os
import pandas as pd

In [11]:
%%writefile constants.py
EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
MODEL_OUTPUT_PATH = '/kaggle/input/qwen3-8b-embedding'
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"

CLEAN_TEXT = True
TOP_K = 2000
BATCH_SIZE = 128

Overwriting constants.py


In [ ]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist

from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm

from constants import CLEAN_TEXT


def build_prompt(row):
    return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""


def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )



def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv").sample(frac=0.6, random_state=42).reset_index(drop=True)

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)


    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: -1,
            }
        )

    return dataframe

Overwriting utils.py


In [ ]:
%%writefile semantic.py
import os
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from peft import PeftModel, PeftConfig

# faiss (GPU優先、失敗時はCPUにフォールバック)
import faiss

from utils import get_dataframe_to_train, prepare_dataframe
from constants import (
    DATA_PATH,
    EMBDEDDING_MODEL_PATH,
    EMBEDDING_MODEL_QUERY,
    TOP_K,
    BATCH_SIZE,
    MODEL_OUTPUT_PATH,
)


def _build_faiss_ip_index(doc_embeddings_np: "np.ndarray"):
    """Build a Faiss IndexFlatIP. Prefer GPU if available.
    doc_embeddings_np: float32, shape [N, D], L2-normalized.
    Returns: (index, is_gpu)
    """
    try:
        # GPU 資源の確保と半精度最適化
        res = faiss.StandardGpuResources()
        cfg = faiss.GpuIndexFlatConfig()
        cfg.useFloat16 = True  # メモリ/速度の最適化
        cfg.device = 0
        index = faiss.GpuIndexFlatIP(res, doc_embeddings_np.shape[1], cfg)
        index.add(doc_embeddings_np)
        return index, True
    except Exception:
        # GPUが使えない/初期化に失敗 → CPU Index
        cpu_index = faiss.IndexFlatIP(doc_embeddings_np.shape[1])
        cpu_index.add(doc_embeddings_np)
        return cpu_index, False


def _faiss_search(index, query_np: "np.ndarray", top_k: int):
    """Run Faiss search and return (D, I). Ensures k <= N."""
    N = index.ntotal
    k = min(top_k, N) if N > 0 else 0
    if k == 0:
        import numpy as np
        return np.empty((len(query_np), 0), dtype="float32"), np.empty((len(query_np), 0), dtype="int64")
    D, I = index.search(query_np, k)
    return D, I


def get_scores(test_dataframe: pd.DataFrame) -> pd.DataFrame:
    # コーパスを作成・前処理
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)

    # 1) LoRA をマージして埋め込みモデルを構築
    model = AutoModelForCausalLM.from_pretrained(EMBDEDDING_MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(EMBDEDDING_MODEL_PATH)

    adapter_config = PeftConfig.from_pretrained(MODEL_OUTPUT_PATH)
    lora_model = PeftModel.from_pretrained(model, MODEL_OUTPUT_PATH, config=adapter_config)
    merged_model = lora_model.merge_and_unload()
    tokenizer.save_pretrained("Qwen3Emb_Finetuned")
    merged_model.save_pretrained("Qwen3Emb_Finetuned")

    # 2) SentenceTransformer としてロード（GPUでencode、結果はnpへ）
    embedding_model = SentenceTransformer(model_name_or_path="Qwen3Emb_Finetuned", device="cuda")

    print("Done loading model!")

    results = []

    # ルール単位でまとめて検索（indexはルールごとに作成）
    for rule in tqdm(test_dataframe["rule"].unique(), desc="FAISS build/search per rule"):
        test_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_part = corpus_part.reset_index(names="row_id")

        # コーパス埋め込み（np.float32, L2正規化済み）
        # convert_to_tensor=False で np.ndarray を受け取り、GPUで推論→CPUへ戻す（Faissに渡しやすい）
        doc_emb = embedding_model.encode(
            sentences=corpus_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=False,
            device="cuda",
            normalize_embeddings=True,  # L2正規化（cosine = 内積）
        )

        # 念のため再度正規化（normalize_embeddings=True だが二重でも安全）
        # Faissはfloat32想定
        import numpy as np
        doc_emb = np.asarray(doc_emb, dtype="float32", order="C")
        if doc_emb.size > 0:
            faiss.normalize_L2(doc_emb)

        # FAISS Index（GPU優先）
        index, used_gpu = _build_faiss_ip_index(doc_emb)

        # クエリ埋め込み（np.float32, L2正規化済み）
        qry_emb = embedding_model.encode(
            sentences=test_part["prompt"].tolist(),
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=False,
            device="cuda",
            normalize_embeddings=True,
        )
        qry_emb = np.asarray(qry_emb, dtype="float32", order="C")
        if qry_emb.size > 0:
            faiss.normalize_L2(qry_emb)

        # Top-K 検索
        D, I = _faiss_search(index, qry_emb, TOP_K)

        # semantic_search と互換の形式へ整形
        semantic_results = []  # List[List[{'corpus_id': int, 'score': float}]]
        for i in range(I.shape[0]):
            items = []
            for j in range(I.shape[1]):
                corpus_id = int(I[i, j])
                score = float(D[i, j])
                items.append({"corpus_id": corpus_id, "score": score})
            semantic_results.append(items)

        test_part["semantic"] = semantic_results

        # 既存ロジックと同じ集計（score × rule_violation を足し合わせ）
        def get_score(semantic):
            semantic = pd.DataFrame(semantic)
            if semantic.empty:
                return 0.0
            semantic = semantic.merge(
                corpus_part[["row_id", "rule_violation"]],
                how="left",
                left_on="corpus_id",
                right_on="row_id",
            )
            semantic["score"] = semantic["score"] * semantic["rule_violation"]
            return float(semantic["score"].sum())

        tqdm.pandas(desc=f"Aggregate scores for rule={rule}")
        test_part["rule_violation"] = test_part["semantic"].progress_apply(get_score)
        results.append(test_part[["row_id", "rule_violation"]].copy())

        # indexはルールごとに破棄（GPUメモリ節約）
        del index

    submission = pd.concat(results, axis=0)
    return submission


def generate_submission():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)

    submission = get_scores(test_dataframe)
    submission = test_dataframe[["row_id"]].merge(submission, on="row_id", how="left")
    submission.to_csv("submission_qwen3.csv", index=False)


if __name__ == "__main__":
    # Kaggle 環境でtokenizersの過剰並列を抑制（任意）
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    generate_submission()


Writing semantic.py


In [14]:
!python semantic.py

## Triplet

In [ ]:
%%writefile triplet.py
#!/usr/bin/env python3

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import random
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    models
)
from sentence_transformers.losses import TripletLoss
from sklearn.metrics.pairwise import cosine_similarity  # (元のまま)
import re
from urllib.parse import urlparse
# import faiss  # (必要なら後で追加)
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Advanced clustering
from sklearn.cluster import AgglomerativeClustering
# from sklearn.cluster import KMeans  # 最小変更のため未使用
# UMAP は停止（条件を厳格化する代わりに完全停止）
# from umap import UMAP

# -----------------------------
# Helpers
# -----------------------------
def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'
    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            if domain.startswith('www.'):
                domain = domain[4:]
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"
    return re.sub(url_pattern, replace_url, str(text))


def load_test_data():
    """Load test data."""
    print("Loading test data...")
    test_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
    print(f"Loaded {len(test_df)} test examples")
    print(f"Unique rules: {test_df['rule'].nunique()}")
    return test_df


def collect_all_texts(test_df):
    """Collect all unique texts from test set."""
    print("\nCollecting all texts for embedding...")
    all_texts = set()
    for body in test_df['body']:
        if pd.notna(body):
            all_texts.add(cleaner(str(body)))
    example_cols = ['positive_example_1', 'positive_example_2',
                    'negative_example_1', 'negative_example_2']
    for col in example_cols:
        for example in test_df[col]:
            if pd.notna(example):
                all_texts.add(cleaner(str(example)))
    all_texts = list(all_texts)
    print(f"Collected {len(all_texts)} unique texts")
    return all_texts


def generate_embeddings(texts, model, batch_size=64):
    """Generate BGE embeddings for all texts."""
    print(f"Generating embeddings for {len(texts)} texts...")
    # device="cuda" を明示し、正規化済みベクトルを得る
    embeddings = model.encode(
        sentences=texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True,
        device="cuda",
    )
    return embeddings


def create_test_triplet_dataset(test_df, augmentation_factor=2, random_seed=42, subsample_fraction=1.0):
    """Create triplet dataset from test data: anchor=rule, positive=positive_example, negative=negative_example."""
    random.seed(random_seed)
    np.random.seed(random_seed)
    anchors, positives, negatives = [], [], []
    print("Creating rule-aligned triplets from test data...")
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test rows"):
        rule = cleaner(str(row['rule']))
        pos_examples = []
        neg_examples = []
        for neg_col in ['negative_example_1', 'negative_example_2']:
            if pd.notna(row[neg_col]):
                pos_examples.append(cleaner(str(row[neg_col])))
        for pos_col in ['positive_example_1', 'positive_example_2']:
            if pd.notna(row[pos_col]):
                neg_examples.append(cleaner(str(row[pos_col])))
        for pos_ex in pos_examples:
            for neg_ex in neg_examples:
                anchors.append(rule)
                positives.append(pos_ex)
                negatives.append(neg_ex)

    if augmentation_factor > 0:
        print(f"Adding {augmentation_factor}x augmentation...")
        rule_positives = {}
        rule_negatives = {}
        for rule in test_df['rule'].unique():
            rule_df = test_df[test_df['rule'] == rule]
            pos_pool, neg_pool = [], []
            for _, row in rule_df.iterrows():
                for neg_col in ['negative_example_1', 'negative_example_2']:
                    if pd.notna(row[neg_col]):
                        pos_pool.append(cleaner(str(row[neg_col])))
                for pos_col in ['positive_example_1', 'positive_example_2']:
                    if pd.notna(row[pos_col]):
                        neg_pool.append(cleaner(str(row[pos_col])))
            rule_positives[rule] = list(set(pos_pool))
            rule_negatives[rule] = list(set(neg_pool))

        for rule in test_df['rule'].unique():
            clean_rule = cleaner(str(rule))
            pos_pool = rule_positives[rule]
            neg_pool = rule_negatives[rule]
            # subsample_fraction で全体を縮退可能
            n_samples = int(min(augmentation_factor * len(pos_pool), max(0, len(pos_pool) * len(neg_pool))) * subsample_fraction)
            for _ in range(n_samples):
                if pos_pool and neg_pool:
                    anchors.append(clean_rule)
                    positives.append(random.choice(pos_pool))
                    negatives.append(random.choice(neg_pool))

    combined = list(zip(anchors, positives, negatives))
    random.shuffle(combined)
    original_count = len(combined)
    if subsample_fraction < 1.0:
        n_samples = int(len(combined) * subsample_fraction)
        combined = combined[:n_samples]
        print(f"Subsampled {original_count} -> {len(combined)} triplets ({subsample_fraction*100:.1f}%)")
    anchors, positives, negatives = zip(*combined) if combined else ([], [], [])
    print(f"Created {len(anchors)} triplets from test data")
    dataset = Dataset.from_dict({'anchor': list(anchors), 'positive': list(positives), 'negative': list(negatives)})
    return dataset


def fine_tune_model(model, train_dataset, epochs=3, batch_size=32, learning_rate=2e-5, margin=0.25, output_dir="./models/test-finetuned-bge"):
    """Fine-tune the sentence transformer model using triplet loss on test data."""
    print(f"Fine-tuning model on {len(train_dataset)} triplets...")
    loss = TripletLoss(model=model, triplet_margin=margin)
    dataset_size = len(train_dataset)
    steps_per_epoch = max(1, dataset_size // batch_size)
    max_steps = steps_per_epoch * epochs
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        warmup_steps=0,
        learning_rate=learning_rate,
        logging_steps=max(1, max_steps // 4),
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        max_grad_norm=1.0,
        dataloader_drop_last=False,
        gradient_checkpointing=False,  # 短時間学習ではオーバーヘッドになりやすい
        gradient_accumulation_steps=1,
        max_steps=max_steps,
        report_to="none"
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_dataset, loss=loss)
    trainer.train()
    final_model_path = f"{output_dir}/final"
    print(f"Saving fine-tuned model to {final_model_path}...")
    model.save_pretrained(final_model_path)
    return model, final_model_path


def load_or_create_finetuned_model(test_df):
    """Prefer using an existing fine-tuned model; otherwise SKIP training and use base model for speed."""
    fine_tuned_path = "./models/test-finetuned-bge/final"
    if os.path.exists(fine_tuned_path):
        print(f"Loading existing fine-tuned model from {fine_tuned_path}...")
        try:
            word_embedding_model = models.Transformer(fine_tuned_path, max_seq_length=128, do_lower_case=True)
            pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
            model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
            print("Loaded fine-tuned model with explicit pooling")
        except Exception:
            model = SentenceTransformer(fine_tuned_path)
            print("Loaded fine-tuned model with default configuration")
        model.half()
        return model

    # ランタイム短縮のため、学習をスキップして軽量ベースモデルを使用
    print("Fine-tuned model not found. Skipping training and using base model for speed...")
    try:
        # Kaggle で利用できる軽量モデルがあれば指定
        model_path = "/kaggle/input/baai/transformers/bge-small-en-v1.5/1"
        word_embedding_model = models.Transformer(model_path, max_seq_length=256, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from Kaggle path (bge-small)")
    except Exception:
        # HF 名称解決にフォールバック（オフライン環境では失敗する可能性あり）
        model_path = "BAAI/bge-small-en-v1.5"
        word_embedding_model = models.Transformer(model_path, max_seq_length=256, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from Hugging Face (bge-small)")

    base_model.half()
    return base_model


def generate_rule_embeddings(test_df, model):
    """Generate embeddings for each unique rule (batched)."""
    print("Generating rule embeddings...")
    unique_rules = pd.Series(test_df['rule'].unique(), dtype=str).apply(lambda x: cleaner(str(x))).tolist()
    rule_emb_matrix = model.encode(unique_rules, convert_to_tensor=False, normalize_embeddings=True, device="cuda")
    rule_embeddings = {rule: emb for rule, emb in zip(test_df['rule'].unique(), rule_emb_matrix)}
    print(f"Generated embeddings for {len(rule_embeddings)} rules")
    return rule_embeddings


# -----------------------------
# UMAP 停止 + クラスタリング頻度削減（最小変更）
# -----------------------------
def create_rule_centroids_with_hierarchical_clustering(test_df, text_to_embedding, rule_embeddings):
    """Create centroids using simplified clustering rules (UMAP disabled)."""
    print(f"\nCreating rule centroids (UMAP disabled, Agglomerative only if enough samples)...")
    rule_centroids = {}

    SMALL_THRESHOLD = 30  # これ以下はクラスタ=1（平均）のみ

    for rule in test_df['rule'].unique():
        rule_data = test_df[test_df['rule'] == rule]

        pos_embeddings, neg_embeddings = [], []
        for _, row in rule_data.iterrows():
            for col in ['positive_example_1', 'positive_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        pos_embeddings.append(text_to_embedding[clean_text])
        for _, row in rule_data.iterrows():
            for col in ['negative_example_1', 'negative_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        neg_embeddings.append(text_to_embedding[clean_text])

        if not pos_embeddings or not neg_embeddings:
            continue

        pos_embeddings = np.array(pos_embeddings)
        neg_embeddings = np.array(neg_embeddings)

        # UMAP 完全停止（そのままの次元でクラスタリング or 平均）
        pos_reduced = pos_embeddings
        neg_reduced = neg_embeddings

        # クラスタ数の決定（小規模なら平均1つ、大きめだけAgglomerative）
        n_pos_clusters = 1 if len(pos_embeddings) <= SMALL_THRESHOLD else min(3, len(pos_embeddings))
        n_neg_clusters = 1 if len(neg_embeddings) <= SMALL_THRESHOLD else min(3, len(neg_embeddings))

        pos_centroids = []
        if n_pos_clusters > 1:
            pos_clusterer = AgglomerativeClustering(n_clusters=n_pos_clusters)
            pos_labels = pos_clusterer.fit_predict(pos_reduced)
            for cluster_id in np.unique(pos_labels):
                cluster_mask = pos_labels == cluster_id
                cluster_embeddings = pos_embeddings[cluster_mask]
                cluster_centroid = cluster_embeddings.mean(axis=0)
                cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)
                pos_centroids.append(cluster_centroid)
        else:
            pos_centroid = pos_embeddings.mean(axis=0)
            pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
            pos_centroids.append(pos_centroid)

        neg_centroids = []
        if n_neg_clusters > 1:
            neg_clusterer = AgglomerativeClustering(n_clusters=n_neg_clusters)
            neg_labels = neg_clusterer.fit_predict(neg_reduced)
            for cluster_id in np.unique(neg_labels):
                cluster_mask = neg_labels == cluster_id
                cluster_embeddings = neg_embeddings[cluster_mask]
                cluster_centroid = cluster_embeddings.mean(axis=0)
                cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)
                neg_centroids.append(cluster_centroid)
        else:
            neg_centroid = neg_embeddings.mean(axis=0)
            neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)
            neg_centroids.append(neg_centroid)

        rule_centroids[rule] = {
            'positive_centroids': pos_centroids,
            'negative_centroids': neg_centroids,
            'pos_count': len(pos_embeddings),
            'neg_count': len(neg_embeddings),
            'rule_embedding': rule_embeddings.get(rule)
        }

        print(f"  Rule: {str(rule)[:50]}... - Pos: {len(pos_embeddings)}, Neg: {len(neg_embeddings)} - Clusters: Pos={len(pos_centroids)}, Neg={len(neg_centroids)}")

    print(f"Created centroids for {len(rule_centroids)} rules")
    return rule_centroids


def predict_test_set_with_hierarchical_clustering(test_df, text_to_embedding, rule_centroids):
    """Vectorized prediction using cosine similarity (normalized embeddings)."""
    print("\nMaking predictions on test set (vectorized matmul)...")
    row_ids_all = []
    preds_all = []

    for rule in test_df['rule'].unique():
        rule_data = test_df[test_df['rule'] == rule]
        if rule not in rule_centroids:
            continue

        pos_centroids = np.array(rule_centroids[rule]['positive_centroids'])  # [P,D]
        neg_centroids = np.array(rule_centroids[rule]['negative_centroids'])  # [N,D]

        # ルールに属する body 埋め込みを一括で行列化
        bodies, row_ids = [], []
        for _, row in rule_data.iterrows():
            body = cleaner(str(row['body']))
            if body in text_to_embedding:
                bodies.append(text_to_embedding[body])
                row_ids.append(row['row_id'])
        if not bodies:
            continue
        bodies = np.array(bodies)  # [B,D], 正規化済み想定

        # コサイン類似度 = 内積（正規化済み）
        # 最小L2距離の差よりも 2*(max_pos_dot - max_neg_dot) を用いる（単調変換で順位は保たれる）
        pos_scores = bodies @ pos_centroids.T  # [B,P]
        neg_scores = bodies @ neg_centroids.T  # [B,N]

        max_pos = pos_scores.max(axis=1)
        max_neg = neg_scores.max(axis=1)
        predictions = 2.0 * (max_pos - max_neg)

        row_ids_all.extend(row_ids)
        preds_all.extend(predictions.tolist())

    print(f"Made predictions for {len(preds_all)} test examples")
    return row_ids_all, np.array(preds_all)


def main():
    print("="*70)
    print("IMPROVED SIMILARITY CLASSIFIER - (UMAP disabled) + Vectorized Inference")
    print("="*70)

    test_df = load_test_data()

    print("\n" + "="*50)
    print("MODEL PREPARATION PHASE")
    print("="*50)
    model = load_or_create_finetuned_model(test_df)

    all_texts = collect_all_texts(test_df)

    print("\n" + "="*50)
    print("EMBEDDING GENERATION PHASE")
    print("="*50)
    all_embeddings = generate_embeddings(all_texts, model)

    text_to_embedding = {text: emb for text, emb in zip(all_texts, all_embeddings)}

    rule_embeddings = generate_rule_embeddings(test_df, model)

    # UMAP 停止 + 小規模はクラスタ=1
    rule_centroids = create_rule_centroids_with_hierarchical_clustering(test_df, text_to_embedding, rule_embeddings)

    print("\n" + "="*50)
    print("PREDICTION PHASE")
    print("="*50)
    row_ids, predictions = predict_test_set_with_hierarchical_clustering(test_df, text_to_embedding, rule_centroids)

    submission_df = pd.DataFrame({'row_id': row_ids, 'rule_violation': predictions})
    submission_df.to_csv('Triplet_submission.csv', index=False)

    print(f"\nSaved predictions for {len(submission_df)} test examples to Triplet_submission.csv")

    print(f"\n{'='*70}")
    print(f"VECTORIZED INFERENCE COMPLETED")
    print(f"Predicted on {len(test_df)} test examples")
    if len(predictions):
        print(f"Prediction stats: min={predictions.min():.4f}, max={predictions.max():.4f}, mean={predictions.mean():.4f}")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()


Writing triplet.py


## Deberta

In [16]:
%%writefile models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel
from transformers.modeling_outputs import SequenceClassifierOutput

class JigsawModelTextW(nn.Module):
    def __init__(self, model_name: str, num_labels: int,  num_freeze_layer: int = 12):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size * 2, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

        if num_freeze_layer > 0:
            self._freeze_top_half_layers(num_freeze_layer)

    def _freeze_top_half_layers(self, num_freeze_layer):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = num_freeze_layer

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def _extract_text_tokens(self, token_type_ids, attention_mask, last_hidden_state):
        """token_type_idsを使用してtext部分を抽出"""
        batch_size = token_type_ids.size(0)
        text_embeddings = []

        for i in range(batch_size):
            # text部分（token_type_id == 1）のマスクを作成
            text_mask = (token_type_ids[i] == 1) & (attention_mask[i] == 1)

            if text_mask.any():
                # text部分の埋め込みを抽出
                text_tokens = last_hidden_state[i][text_mask]
                text_pooled = text_tokens.mean(dim=0)
            else:
                text_pooled = torch.zeros(self.config.hidden_size, device=token_type_ids.device)

            text_embeddings.append(text_pooled)

        return torch.stack(text_embeddings)


    def _pool_text_only(self, token_type_ids, attention_mask, last_hidden_state):
        # last_hidden_state: [B,S,H], attention_mask: [B,S], token_type_ids: [B,S] or None
        use_type_ids = (
            token_type_ids is not None
            and token_type_ids.dim() == 2
            and torch.any(token_type_ids > 0)
        )
        if use_type_ids:
            mask_2d = (token_type_ids == 1) & (attention_mask == 1)  # [B,S] boolean AND
        else:
            raise Exception

        mask = mask_2d.unsqueeze(-1).type_as(last_hidden_state)      # [B,S,1] -> broadcast OK
        num = (last_hidden_state * mask).sum(dim=1)                  # [B,H]
        den = mask.sum(dim=1).clamp(min=1e-9)                        # [B,1]
        return num / den

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        sequence_output, _ = outputs.last_hidden_state.max(1)  # tuple: [layer0..last]

        # Text部分のみのpooling
        text_features = self._pool_text_only(
            token_type_ids, attention_mask, outputs.last_hidden_state
        )  # [batch_size, hidden_size]

        # concat
        sequence_output = torch.cat([sequence_output, text_features], dim=1)
        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)


class JigsawModelConcatrate(nn.Module):
    def __init__(self, model_name: str, num_labels: int, num_freeze_layer = 12):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size * 4, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

        if num_freeze_layer > 0:
            self._freeze_top_half_layers()

    def _freeze_top_half_layers(self,  num_freeze_layer):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = num_freeze_layer

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        sequence_output = torch.cat([outputs["last_hidden_state"][-1*i][:,0] for i in range(1, 4+1)], dim=1)  # concatenate
        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

class JigsawModelCustomHeader(nn.Module):
    def __init__(self, model_name: str, header:str, num_labels: int,  num_freeze_layer: int = 12):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "num_labels": num_labels})
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.regressor = nn.Linear(self.config.hidden_size, num_labels)
        self.header = header

        if self.header == "lstm":
            self.lstm = nn.LSTM(self.config.hidden_size, self.config.hidden_size, batch_first=True)
        elif self.header == "attention":
            # Attention layer
            self.attention = nn.MultiheadAttention(
                embed_dim=self.config.hidden_size,
                num_heads=8,  # 通常8または16
                dropout=0.1,
                batch_first=True  # 重要: batch_first=True
            )

            # Layer normalization and dropout
            self.layer_norm = nn.LayerNorm(self.config.hidden_size)
            self.dropout = nn.Dropout(0.1)

        # 共通
        self.regressor = nn.Linear(self.config.hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()


        if  num_freeze_layer > 0:
            self._freeze_top_half_layers(num_freeze_layer)

    def _freeze_top_half_layers(self, num_freeze_layer: int = 12):
        """先頭半分の層をfreeze"""
        n_layers = self.config.num_hidden_layers
        freeze_count = num_freeze_layer

        print(f"Freezing top {freeze_count} layers out of {n_layers} total layers")

        for i in range(freeze_count):
            for name, param in self.backbone.encoder.layer[i].named_parameters():
                param.requires_grad = False

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,  # DeBERTa/RoBERTaはNoneでOK
        labels=None
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,  # 念のため明示
            return_dict=True
        )
        if self.header == "maxpooling":
            sequence_output, _ = outputs['last_hidden_state'].max(1)  # max pooling
        elif self.header == "meanpooling":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs['last_hidden_state'].size()).float()
            sequence_output = torch.sum(outputs['last_hidden_state'] * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        elif self.header == "lstm":
            out, _ = self.lstm(outputs['last_hidden_state'], None)
            sequence_output = out[:, -1, :]
        elif self.header == "attention":
            # 最後の隠れ状態を取得
            last_hidden_state = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

            # Self-attention適用
            # key_padding_maskでPADトークンをマスク
            key_padding_mask = ~attention_mask.bool()  # PADトークンをTrue

            attn_output, attn_weights = self.attention(
                query=last_hidden_state,
                key=last_hidden_state,
                value=last_hidden_state,
                key_padding_mask=key_padding_mask
            )

            # Residual connection + Layer Norm
            attn_output = self.layer_norm(attn_output + last_hidden_state)
            attn_output = self.dropout(attn_output)

            # [CLS]トークン（最初のトークン）を取得
            sequence_output = attn_output[:, 0, :]  # [batch_size, hidden_size]

        logits = self.regressor(sequence_output)

        loss = None
        if labels is not None:
            # 分類（0/1）のときはCrossEntropy
            loss = self.loss_fn(logits, labels.long())

        return SequenceClassifierOutput(loss=loss, logits=logits)

Writing models.py


In [ ]:
%%writefile deberta.py
import os
import re
import pandas as pd
import numpy as np
import random
import torch
from urllib.parse import urlparse
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

from models import *
import os

os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "9994"  # modify if RuntimeError: Address already in use
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"

def url_to_semantics(text):
    if not isinstance(text, str):
        return ""
    urls = re.findall(r'https?://[^\s/$.?#].[^\s]*', text)
    if not urls:
        return ""
    all_semantics = []
    seen = set()
    for url in urls:
        u = url.lower()
        m = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", u)
        if m:
            parts = m.group(1).split(".")
            for p in parts:
                if p and len(p) > 3 and p not in seen:
                    all_semantics.append(f"domain:{p}")
                    seen.add(p)
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", u)
        parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()]
        for p in parts:
            pc = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", p)
            if pc and len(pc) > 3 and pc not in seen:
                all_semantics.append(f"path:{pc}")
                seen.add(pc)
    if not all_semantics:
        return ""
    return "\nURL Keywords: " + " ".join(all_semantics)

def get_dataframe_to_train(data_path):
    train_df = pd.read_csv(f"{data_path}/train.csv")
    test_df = pd.read_csv(f"{data_path}/test.csv")
    out = []
    for k in ["positive","negative"]:
        for i in range(1,3):
            c = f"{k}_example_{i}"
            if c in train_df.columns:
                sub = train_df[[c,"rule","subreddit"]].copy()
                sub = sub.rename(columns={c:"body"})
                sub["rule_violation"] = 1 if k=="positive" else 0
                sub = sub.dropna(subset=["body"])
                sub = sub[sub["body"].str.strip().str.len()>0]
                if len(sub):
                    out.append(sub)
    for k in ["positive","negative"]:
        for i in range(1,3):
            c = f"{k}_example_{i}"
            if c in test_df.columns:
                sub = test_df[[c,"rule","subreddit"]].copy()
                sub = sub.rename(columns={c:"body"})
                sub["rule_violation"] = 1 if k=="positive" else 0
                sub = sub.dropna(subset=["body"])
                sub = sub[sub["body"].str.strip().str.len()>0]
                if len(sub):
                    out.append(sub)
    df = pd.concat(out, axis=0) if out else pd.DataFrame(columns=["body","rule","subreddit","rule_violation"])
    df = df.drop_duplicates(subset=["body","rule","subreddit"], ignore_index=True)
    df = df.drop_duplicates(subset=["body","rule"], keep="first")
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

def seed_everything(s=42):
    random.seed(s)
    os.environ["PYTHONHASHSEED"]=str(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

class CFG:
    model_name_or_path="/kaggle/input/huggingfacedebertav3variants/deberta-v3-large-offensive"
    data_path="/kaggle/input/jigsaw-agile-community-rules/"
    output_dir="./deberta_v3_small_final_model"
    EPOCHS=3
    LEARNING_RATE=2e-5
    MAX_LENGTH=256
    BATCH_SIZE=4
    N_SPLITS=5
    RANDOM_STATE=42

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings=encodings
        self.labels=labels
    def __getitem__(self, idx):
        item={k:torch.tensor(v[idx]) for k,v in self.encodings.items()}
        if self.labels is not None:
            item["labels"]=torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings["input_ids"])

def build_inputs(df):
    df=df.copy()
    df["body_with_url"]=df["body"].apply(lambda x: x+url_to_semantics(x))
    df["input_text"]=df["rule"]+"[SEP]"+df["body_with_url"]
    return df

def tokenize(tokenizer, texts, max_len):
    return tokenizer(texts, truncation=True, padding=True, max_length=max_len)

def train_and_predict_submission():
    seed_everything(42)
    df=get_dataframe_to_train(CFG.data_path)
    df=build_inputs(df)
    test_df=pd.read_csv(f"{CFG.data_path}/test.csv")
    test_df=build_inputs(test_df)
    tokenizer=AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    tr_enc=tokenize(tokenizer, df["input_text"].tolist(), CFG.MAX_LENGTH)
    tr_lbl=df["rule_violation"].tolist()
    tr_ds=JigsawDataset(tr_enc, tr_lbl)

    model=AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
    # model = JigsawModelTextW(CFG.model_name_or_path, num_labels=2, num_freeze_layer=13)

    args=TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.EPOCHS,
        learning_rate=CFG.LEARNING_RATE,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        # gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        report_to="wandb",
        run_name="deberta-v3-large-deepspeed-zero3",
        deepspeed="ds_config_zero3.json",
        save_strategy="no",
        # fp16=True,
        logging_steps=10
    )
    trainer=Trainer(model=model, args=args, train_dataset=tr_ds)
    trainer.train()
    te_enc=tokenize(tokenizer, test_df["input_text"].tolist(), CFG.MAX_LENGTH)
    te_ds=JigsawDataset(te_enc, None)
    preds=trainer.predict(te_ds)
    probs=torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=1)[:,1].numpy()
    sub=pd.DataFrame({"row_id":test_df["row_id"],"rule_violation":probs})
    sub.to_csv("deberta_submission.csv", index=False)

def evaluate_fold(trainer, ds, y_true):
    out=trainer.predict(ds)
    probs=torch.nn.functional.softmax(torch.tensor(out.predictions), dim=1)[:,1].numpy()
    preds=out.predictions.argmax(axis=1).astype(int)
    p,r,f1,_=precision_recall_fscore_support(y_true, preds, average="binary")
    auc=roc_auc_score(y_true, probs)
    return dict(precision=p, recall=r, f1=f1, auc=auc)

def cross_validation():
    seed_everything(CFG.RANDOM_STATE)
    df=get_dataframe_to_train(CFG.data_path)
    df=build_inputs(df)
    skf=StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)
    tokenizer=AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    scores=[]
    for i,(tr_idx,va_idx) in enumerate(skf.split(df, df["rule"]),1):
        tr=df.iloc[tr_idx].reset_index(drop=True)
        va=df.iloc[va_idx].reset_index(drop=True)
        tr_enc=tokenize(tokenizer, tr["input_text"].tolist(), CFG.MAX_LENGTH)
        va_enc=tokenize(tokenizer, va["input_text"].tolist(), CFG.MAX_LENGTH)
        tr_ds=JigsawDataset(tr_enc, tr["rule_violation"].tolist())
        va_ds=JigsawDataset(va_enc, va["rule_violation"].tolist())
        model=AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
        # model = JigsawModelTextW(CFG.model_name_or_path, num_labels=2, num_freeze_layer=13)

        args=TrainingArguments(
            output_dir=CFG.output_dir,
            num_train_epochs=CFG.EPOCHS,
            # learning_rate=CFG.LEARNING_RATE,
            # per_device_train_batch_size=CFG.BATCH_SIZE,
            # gradient_accumulation_steps=4,
            # warmup_ratio=0.1,
            # weight_decay=0.01,
            report_to="none",
            # run_name="deberta-v3-large-deepspeed-zero3",
            deepspeed="ds_config_zero3.json",
            save_strategy="no",
            # fp16=True,
            logging_steps=10
        )
        trainer=Trainer(model=model, args=args, train_dataset=tr_ds)
        trainer.train()
        res=evaluate_fold(trainer, va_ds, va["rule_violation"].values)
        scores.append(res)
        print(f"Fold {i} AUC={res['auc']:.4f} F1={res['f1']:.4f} P={res['precision']:.4f} R={res['recall']:.4f}")
    aucs=[s["auc"] for s in scores]
    print(f"AUC-ROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

if __name__=="__main__":
    mode=os.getenv("RUN_MODE","cv").lower()
    if mode=="train":
        train_and_predict_submission()
    else:
        cross_validation()

Overwriting deberta.py


In [18]:
!python triplet.py

2025-10-22 15:09:39.491712: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761145779.706234     223 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761145779.767466     223 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
IMPROVED SIMILARITY CLASSIFIER - HIERARCHICAL CLUSTERING + UMAP (same results as your script)
Loading test data...
Loaded 10 test examples
Unique rules: 2

MODEL PREPARATION PHASE
Fine-tuned model not found. Creating new one...
Loading base BGE embedding model...
Loaded base model from Kaggle path with explicit pooling
Creating rule-aligned triplets from test data...
Processing test rows: 100%|███████████████████| 10/10 [00:00<00:00,

In [23]:
%env RUN_MODE=train
!python deberta.py

env: RUN_MODE=cv
2025-10-22 15:17:39.752155: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761146259.774406     439 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761146259.781077     439 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[2025-10-22 15:17:46,535] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-22 15:17:47,890] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
  0%|                                                   | 0/300 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning:

In [20]:
import pandas as pd
import numpy as np

# --------- 읽기 ---------
deb = pd.read_csv("deberta_submission.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "deb"})
tri = pd.read_csv("Triplet_submission.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "tri"})
q   = pd.read_csv("submission_qwen.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "q"})
q3  = pd.read_csv("submission_qwen3.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "q3"})

# --------- 교집합 머지 ---------
dfs = [deb, tri, q, q3]
df = dfs[0]
for d in dfs[1:]:
    df = df.merge(d, on="row_id", how="inner")

def minmax_scale(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    fill = s.median()
    s = s.fillna(0.5 if pd.isna(fill) else fill)
    mn, mx = s.min(), s.max()
    if mx > mn:
        return (s - mn) / (mx - mn)
    return pd.Series(0.5, index=s.index, dtype=float)  # 상수 벡터면 중립값

# --------- 모델별 스케일링 ---------
for col in ["deb", "tri", "q", "q3"]:
    df[f"{col}_s"] = minmax_scale(df[col])

# --------- 가중치(필요시 수정) ---------
weights = {
    "deb_s": 0.43,
    "tri_s": 0.27,
    "q_s"  : 0.22,
    "q3_s" : 0.08,
}

# 안전장치: 합 1이 아니면 정규화
wsum = sum(weights.values())
if abs(wsum - 1.0) > 1e-8:
    weights = {k: v / wsum for k, v in weights.items()}

# --------- 앙상블 ---------
df["rule_violation"] = sum(weights[col] * df[col] for col in weights.keys())

# --------- 저장 ---------
out_cols = ["row_id", "rule_violation"]
df[out_cols].to_csv("submission.csv", index=False)

# 참고용 상관관계(스케일 후) 출력
print("Scaled predictions correlation:\n", df[[k for k in weights.keys()]].corr())
print(f"submission.csv saved with {len(df)} rows",
      f"(weights: {', '.join([f'{k}:{v:.2f}' for k,v in weights.items()])})")

FileNotFoundError: [Errno 2] No such file or directory: 'deberta_submission.csv'

In [ ]:
cat submission.csv